In [4]:
from data_procesing import load_raw_data, EOSDataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')

raw_test = load_raw_data('test')
test_set = EOSDataset(raw_test,tokenizer)
test_loader = DataLoader(test_set,batch_size=16,shuffle='True')

Reading : en_pud-ud-test.sent_split
Reading : en_partut-ud-test.sent_split
Reading : it_vit-ud-test.sent_split
Reading : en_gum-ud-test.sent_split
Reading : it_partut-ud-test.sent_split
Reading : it_markit-ud-test.sent_split
Reading : en_ewt-ud-test.sent_split
Reading : it_isdt-ud-test.sent_split


In [6]:
import torch

model_name = "bert-base-multilingual-cased"
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=2)
model.classifier.load_state_dict(torch.load("eos_head.pt"))
model.eval()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4339.33it/s]
BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

In [7]:
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        outputs = model(
            input_ids=      batch["input_ids"],
            attention_mask= batch["attention_mask"],
        )
        preds = outputs.logits.argmax(dim=-1)  # (batch, seq_len)

        # Flatten et filtre les -100
        mask = batch["labels"] != -100
        all_preds.extend(preds[mask].tolist())
        all_true.extend(batch["labels"][mask].tolist())

In [8]:
from sklearn.metrics import classification_report
print(classification_report(all_true, all_preds, target_names=["non-EOS", "EOS"]))

              precision    recall  f1-score   support

     non-EOS       1.00      0.98      0.99      5911
         EOS       0.61      0.99      0.76       201

    accuracy                           0.98      6112
   macro avg       0.81      0.98      0.87      6112
weighted avg       0.99      0.98      0.98      6112

